# 01 — Text Preprocessing & Dataset Preparation

**Proyek:** Tujuan Semata — klasifikasi `Label_Intent` pada pesan chat pelanggan
**PIC:** Bayu

Notebook ini menyiapkan data sebelum masuk ke pelatihan model (Main Quest Gen AI #2, #3 & Side Quest Class Balance).

**Alur:**
1. Memuat dataset dan mengambil kolom teks + `Label_Intent`
2. Cek keseimbangan kelas (awal)
3. Pipeline text preprocessing (case folding, hapus URL/emoji/tanda baca, normalisasi slang, tokenisasi)
4. Cek & buang duplikat (supaya tidak ada *data leakage*)
5. Stratified split 80% train / 10% val / 10% test
6. Oversampling kelas timpang (**hanya di train**)
7. Simpan output: `train_intent.csv`, `val_intent.csv`, `test_intent.csv`, `kamus_slang_tujuan_semata.json`, dan grafik distribusi kelas

**Cara pakai:** letakkan `ulasan_dan_chat_tujuan_semata.csv` di `data/processed/` (atau satu folder dengan notebook ini), lalu *Run All*. Di Google Colab, kalau file belum ada, nanti muncul tombol upload.

Library yang dipakai: `pandas`, `numpy`, `scikit-learn`, `matplotlib`.

## 1. Setup & Konfigurasi

In [ ]:
import json
import random
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# ====== KONFIGURASI (ubah di sini kalau perlu) ======
SEED = 42
DATA_PATH = Path("data/processed/ulasan_dan_chat_tujuan_semata.csv")
OUT_DIR = Path("data/processed")   # tempat train/val/test + kamus slang
FIG_DIR = Path("assets")           # tempat grafik untuk README.md

TEXT_COL = "Pesan_Chat"            # sumber teks mentah (bisa diganti "Pesan_Chat_NLP")
LABEL_COL = "Label_Intent"

TEST_SIZE = 0.10
VAL_SIZE = 0.10
IMBALANCE_THRESHOLD = 1.2          # rasio kelas terbesar/terkecil; di atas ini -> oversampling (train saja)

# Opsional: tambahkan sampel out_of_context manual kalau kelas ini dirasa kurang.
# Contoh: EXTRA_OOC_SAMPLES = ["Siapa pemenang piala dunia 2018?", "Tolong buatkan puisi tentang hujan"]
EXTRA_OOC_SAMPLES = []

random.seed(SEED)
np.random.seed(SEED)
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 2. Pemuatan Dataset

Hanya kolom teks dan `Label_Intent` yang diambil. Kolom `Pesan_Chat_NLP` (kalau ada) ikut dibawa **hanya sebagai pembanding** hasil preprocessing di bawah.

In [ ]:
def cari_dataset():
    kandidat = [
        DATA_PATH,
        Path("ulasan_dan_chat_tujuan_semata.csv"),
        Path("/content/ulasan_dan_chat_tujuan_semata.csv"),
    ]
    for p in kandidat:
        if p.exists():
            return p
    try:  # Google Colab: minta upload manual
        from google.colab import files
        print("Dataset belum ditemukan, silakan upload CSV-nya...")
        nama = next(iter(files.upload()))
        return Path(nama)
    except ImportError:
        raise FileNotFoundError(
            "CSV tidak ditemukan. Taruh di data/processed/ atau satu folder dengan notebook ini, "
            "atau ubah DATA_PATH."
        )


csv_path = cari_dataset()
df_raw = pd.read_csv(csv_path)
print(f"Dataset: {csv_path} -> {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")

assert TEXT_COL in df_raw.columns and LABEL_COL in df_raw.columns, "Kolom teks/label tidak ditemukan"

kolom_ambil = list(dict.fromkeys([TEXT_COL, "Pesan_Chat_NLP", LABEL_COL]))
kolom_ambil = [c for c in kolom_ambil if c in df_raw.columns]
df = df_raw[kolom_ambil].dropna(subset=[TEXT_COL, LABEL_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

# Tambahan sampel out_of_context (opsional, lihat konfigurasi)
if EXTRA_OOC_SAMPLES:
    extra = pd.DataFrame({TEXT_COL: EXTRA_OOC_SAMPLES, LABEL_COL: "out_of_context"})
    df = pd.concat([df, extra], ignore_index=True)
    print(f"Ditambahkan {len(extra)} sampel out_of_context manual")

df.head()

## 3. Pemeriksaan Keseimbangan Kelas (Awal)

Rasio = jumlah kelas terbesar / jumlah kelas terkecil. Kalau di atas `IMBALANCE_THRESHOLD`, kelas dianggap timpang.

In [ ]:
def cek_balance(d, judul):
    c = d[LABEL_COL].value_counts().sort_index()
    rasio = c.max() / c.min()
    status = "SEIMBANG" if rasio <= IMBALANCE_THRESHOLD else "TIMPANG"
    print(f"{judul}\n{c.to_string()}\nTotal: {c.sum()} | Rasio terbesar/terkecil: {rasio:.2f} -> {status}\n")
    return c


dist_awal = cek_balance(df, "Distribusi awal")

## 4. Pipeline Text Preprocessing

Urutan langkah:
1. **Case folding** — semua huruf jadi kecil
2. **Hapus URL**
3. **Lindungi istilah teknis** (`c++` -> `cpp`, `c#` -> `csharp`) supaya tidak hilang saat tanda baca dibuang
4. **Hapus emoji**
5. **Hapus tanda baca** (angka dipertahankan karena penting, misalnya jumlah pesanan atau nomor meja)
6. **Rapikan huruf berulang** (`baaanget` -> `baanget`) dan spasi ganda
7. **Tokenisasi** (pisah per kata)
8. **Normalisasi slang** pakai `KAMUS_SLANG` (`gak` -> `tidak`, `rekomen` -> `rekomendasi`, dst.)

> Stopword **sengaja tidak dibuang**: kata tanya seperti *apa, berapa, bagaimana* justru penting untuk membedakan intent.

Kamus slang bisa ditambah sendiri di bawah — kamus final akan disimpan ke JSON.

In [ ]:
KAMUS_SLANG = {
    # kata ganti orang pertama
    "gw": "saya", "gue": "saya", "gua": "saya", "aku": "saya", "ane": "saya",
    # negasi
    "gak": "tidak", "nggak": "tidak", "ngga": "tidak", "ga": "tidak",
    "enggak": "tidak", "engga": "tidak", "tdk": "tidak",
    # kata umum
    "aja": "saja", "doang": "saja",
    "banget": "sekali", "bgt": "sekali",
    "min": "admin",
    "makasih": "terima kasih", "makasi": "terima kasih", "mksh": "terima kasih",
    "gimana": "bagaimana", "gmn": "bagaimana",
    "udah": "sudah", "udh": "sudah", "sdh": "sudah",
    "lg": "lagi", "yg": "yang", "dgn": "dengan", "dg": "dengan", "utk": "untuk",
    "krn": "karena", "karna": "karena", "tp": "tapi", "jgn": "jangan",
    "kalo": "kalau", "klo": "kalau", "kayak": "seperti", "kyk": "seperti",
    "emang": "memang", "pake": "pakai", "pakek": "pakai",
    # istilah domain (kedai)
    "rekomen": "rekomendasi", "reko": "rekomendasi", "rekom": "rekomendasi",
    "pesen": "pesan", "mesen": "pesan",
}

ISTILAH_TEKNIS = {"c++": "cpp", "c#": "csharp"}

URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"   # simbol, emoji, ekspresi
    "\U00002600-\U000027BF"   # misc symbols & dingbats
    "\U0001F000-\U0001F2FF"
    "\U0000FE0F\U0000200D"    # variation selector & zero-width joiner
    "]+",
    flags=re.UNICODE,
)
PUNCT_RE = re.compile(r"[^a-z0-9\s]")
REPEAT_RE = re.compile(r"(.)\1{2,}")


def case_folding(t):
    return t.lower()


def hapus_url(t):
    return URL_RE.sub(" ", t)


def lindungi_istilah_teknis(t):
    for k, v in ISTILAH_TEKNIS.items():
        t = t.replace(k, v)
    return t


def hapus_emoji(t):
    return EMOJI_RE.sub(" ", t)


def hapus_tanda_baca(t):
    return PUNCT_RE.sub(" ", t)


def rapikan(t):
    t = REPEAT_RE.sub(r"\1", t)          # "baaanget" -> "baanget"
    return re.sub(r"\s+", " ", t).strip()


def tokenisasi(t):
    return t.split()


def normalisasi_slang(tokens):
    hasil = [KAMUS_SLANG.get(tok, tok) for tok in tokens]
    return " ".join(hasil).split()       # split ulang karena ada yang jadi 2 kata ("terima kasih")


def preprocess(teks):
    t = case_folding(teks)
    t = hapus_url(t)
    t = lindungi_istilah_teknis(t)
    t = hapus_emoji(t)
    t = hapus_tanda_baca(t)
    t = rapikan(t)
    return normalisasi_slang(tokenisasi(t))


# Uji cepat
for s in ["Min, gw mau pesen 2 Croffle (dine-in) meja no. 4 ya 😊🙏",
          "Tolong buatkan kode Python & C++ buat CNN, makasih!",
          "Gak ada rekomen kopi yg seger ya kak?"]:
    print(s, "\n  ->", preprocess(s))

### Terapkan ke seluruh dataset

In [ ]:
df["Teks_Bersih"] = df[TEXT_COL].apply(lambda s: " ".join(preprocess(s)))
df["Jumlah_Token"] = df["Teks_Bersih"].str.split().str.len()

# Buang baris yang kosong setelah preprocessing (kalau ada)
kosong = (df["Jumlah_Token"] == 0).sum()
if kosong:
    print(f"Membuang {kosong} baris yang kosong setelah preprocessing")
    df = df[df["Jumlah_Token"] > 0].reset_index(drop=True)

print("Statistik panjang teks (jumlah token):")
print(df["Jumlah_Token"].describe().round(1).to_string())

print("\nContoh sebelum -> sesudah:")
for _, r in df.sample(6, random_state=SEED).iterrows():
    print(f"[{r[LABEL_COL]}]")
    print("  RAW  :", r[TEXT_COL])
    print("  CLEAN:", r["Teks_Bersih"])

### (Opsional) Bandingkan dengan kolom `Pesan_Chat_NLP` bawaan dataset

Hanya sanity check. Perbedaan wajar muncul karena kamus slang di notebook ini mungkin tidak sama persis dengan yang dipakai saat membuat kolom bawaan.

In [ ]:
if TEXT_COL != "Pesan_Chat_NLP" and "Pesan_Chat_NLP" in df.columns:
    cocok = (df["Teks_Bersih"] == df["Pesan_Chat_NLP"]).mean()
    print(f"Hasil sama persis dengan Pesan_Chat_NLP: {cocok:.1%}")
    beda = df[df["Teks_Bersih"] != df["Pesan_Chat_NLP"]]
    for _, r in beda.head(5).iterrows():
        print("\n  Notebook :", r["Teks_Bersih"])
        print("  Bawaan   :", r["Pesan_Chat_NLP"])
else:
    print("Pembanding tidak tersedia / tidak relevan, dilewati.")

## 5. Cek Duplikat (pencegah *data leakage*)

Banyak pesan di dataset ini identik (atau identik setelah dibersihkan). Kalau dibiarkan, teks yang sama bisa jatuh ke train **dan** test sekaligus, sehingga skor evaluasi jadi terlalu bagus dan menyesatkan.

Yang dilakukan:
- Teks yang sama tapi **label berbeda** (ambigu) dibuang
- Teks yang sama dan label sama dipertahankan satu saja

In [ ]:
n_sebelum = len(df)

# 1) Teks yang sama tapi label beda
n_label = df.groupby("Teks_Bersih")[LABEL_COL].nunique()
teks_konflik = n_label[n_label > 1].index
print(f"Teks dengan label bentrok: {len(teks_konflik)}")
if len(teks_konflik):
    df = df[~df["Teks_Bersih"].isin(teks_konflik)]

# 2) Duplikat biasa
n_dup = df.duplicated("Teks_Bersih").sum()
df = df.drop_duplicates("Teks_Bersih", keep="first").reset_index(drop=True)

print(f"Duplikat dibuang: {n_dup}")
print(f"Baris: {n_sebelum} -> {len(df)} ({len(df) / n_sebelum:.1%} tersisa)\n")
dist_dedup = cek_balance(df, "Distribusi setelah dedup")

## 6. Stratified Train / Validation / Test Split (80 / 10 / 10)

Pembagian dilakukan **sebelum** oversampling, supaya baris hasil oversampling tidak bocor ke validation/test.

In [ ]:
train_val, test = train_test_split(
    df, test_size=TEST_SIZE, stratify=df[LABEL_COL], random_state=SEED
)
train, val = train_test_split(
    train_val,
    test_size=VAL_SIZE / (1 - TEST_SIZE),   # 10% dari total = 1/9 dari sisa 90%
    stratify=train_val[LABEL_COL],
    random_state=SEED,
)

for nama, d in [("train", train), ("val", val), ("test", test)]:
    print(f"{nama:<5}: {len(d):>5} baris ({len(d) / len(df):.1%})")

# Pastikan tidak ada teks yang sama di dua split berbeda
set_train, set_val, set_test = (set(d["Teks_Bersih"]) for d in (train, val, test))
assert not (set_train & set_val), "Leakage: train & val"
assert not (set_train & set_test), "Leakage: train & test"
assert not (set_val & set_test), "Leakage: val & test"
print("\nTidak ada overlap teks antar split.")

## 7. Penanganan Kelas Timpang (Oversampling — hanya train)

Kalau rasio kelas di train melebihi `IMBALANCE_THRESHOLD`, kelas minoritas di-*oversample* (random, dengan pengembalian) sampai jumlahnya sama dengan kelas terbesar. Validation dan test **tidak** disentuh supaya tetap mencerminkan data asli.

In [ ]:
def oversample(d, seed=SEED):
    target = d[LABEL_COL].value_counts().max()
    bagian = []
    for _, g in d.groupby(LABEL_COL):
        bagian.append(g.sample(target, replace=True, random_state=seed) if len(g) < target else g)
    return pd.concat(bagian).sample(frac=1, random_state=seed).reset_index(drop=True)


dist_train_sebelum = cek_balance(train, "Distribusi train (sebelum oversampling)")
rasio_train = dist_train_sebelum.max() / dist_train_sebelum.min()

if rasio_train > IMBALANCE_THRESHOLD:
    train_final = oversample(train)
    print("Oversampling diterapkan pada train.\n")
else:
    train_final = train.reset_index(drop=True)
    print("Kelas sudah cukup seimbang, oversampling dilewati.\n")

dist_train_final = cek_balance(train_final, "Distribusi train (final)")
val = val.reset_index(drop=True)
test = test.reset_index(drop=True)
print("Val & test (tidak di-oversample):")
print(pd.DataFrame({"val": val[LABEL_COL].value_counts(), "test": test[LABEL_COL].value_counts()}).sort_index().to_string())

## 8. Simpan Output

In [ ]:
KOLOM_OUTPUT = [TEXT_COL, "Teks_Bersih", "Jumlah_Token", LABEL_COL]

train_final[KOLOM_OUTPUT].to_csv(OUT_DIR / "train_intent.csv", index=False)
val[KOLOM_OUTPUT].to_csv(OUT_DIR / "val_intent.csv", index=False)
test[KOLOM_OUTPUT].to_csv(OUT_DIR / "test_intent.csv", index=False)

with open(OUT_DIR / "kamus_slang_tujuan_semata.json", "w", encoding="utf-8") as f:
    json.dump(KAMUS_SLANG, f, ensure_ascii=False, indent=2)

print("Tersimpan di", OUT_DIR.resolve())
for nama in ["train_intent.csv", "val_intent.csv", "test_intent.csv", "kamus_slang_tujuan_semata.json"]:
    print(" -", nama)

## 9. Grafik Distribusi Kelas (untuk README.md)

In [ ]:
def bar_h(ax, counts, judul):
    counts = counts.sort_index()
    bars = ax.barh(counts.index, counts.values)
    ax.bar_label(bars, padding=3, fontsize=9)
    ax.set_title(judul, fontsize=10)
    ax.invert_yaxis()
    ax.set_xlim(0, counts.max() * 1.18)
    ax.set_xlabel("Jumlah sampel")


# Grafik 1: sebelum / sesudah dedup / train final
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
bar_h(axes[0], dist_awal, f"Awal (n={dist_awal.sum()})")
bar_h(axes[1], dist_dedup, f"Setelah dedup (n={dist_dedup.sum()})")
bar_h(axes[2], dist_train_final, f"Train final (n={dist_train_final.sum()})")
fig.suptitle("Distribusi Label_Intent", fontsize=12)
fig.tight_layout()
fig.savefig(FIG_DIR / "distribusi_kelas.png", dpi=150, bbox_inches="tight")
plt.show()

# Grafik 2: distribusi per split
split_df = pd.DataFrame({
    "train (final)": train_final[LABEL_COL].value_counts(),
    "val": val[LABEL_COL].value_counts(),
    "test": test[LABEL_COL].value_counts(),
}).sort_index()
ax = split_df.plot(kind="barh", figsize=(9, 4.5), width=0.8)
ax.invert_yaxis()
ax.set_title("Distribusi Label_Intent per split")
ax.set_xlabel("Jumlah sampel")
plt.tight_layout()
plt.savefig(FIG_DIR / "distribusi_split.png", dpi=150, bbox_inches="tight")
plt.show()

print("Grafik tersimpan di", FIG_DIR.resolve())

## 10. Verifikasi Akhir

In [ ]:
for nama in ["train_intent.csv", "val_intent.csv", "test_intent.csv"]:
    d = pd.read_csv(OUT_DIR / nama)
    print(f"{nama:<18} {d.shape[0]:>5} baris | kelas: {d[LABEL_COL].nunique()} | null: {int(d.isna().sum().sum())}")

kamus = json.load(open(OUT_DIR / "kamus_slang_tujuan_semata.json", encoding="utf-8"))
print(f"kamus_slang_tujuan_semata.json: {len(kamus)} entri")
print("\nSelesai. Dataset siap dipakai untuk notebook pelatihan model.")